In [2]:
import json

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from torch import nn

complaints = pd.read_csv("data/complaints.csv")

complaints["Date received"] = pd.to_datetime(
    complaints["Date received"],
    errors="coerce",
)

complaints["Consumer complaint narrative"] = (
    complaints["Consumer complaint narrative"]
    .fillna("")
    .str.strip()
)

train = complaints[
    complaints["Date received"] < "2023-10-01"
].copy()

validation = complaints[
    (complaints["Date received"] >= "2023-10-01")
    & (complaints["Date received"] < "2023-12-01")
].copy()

test = complaints[
    complaints["Date received"] >= "2023-12-01"
].copy()

print(len(train), len(validation), len(test))


def scores(y_true, predictions, weights=None):
    accuracy = accuracy_score(
        y_true,
        predictions,
        sample_weight=weights,
    )
    macro_f1 = f1_score(
        y_true,
        predictions,
        average="macro",
        sample_weight=weights,
    )
    return round(accuracy, 3), round(macro_f1, 3)


embeddings = np.load("data/complaint-embeddings.npy").astype("float32")

teams = sorted(train["routing_team"].unique())
team_number = {team: number for number, team in enumerate(teams)}


def to_tensors(split):
    X = torch.from_numpy(embeddings[split.index])
    y = torch.tensor(split["routing_team"].map(team_number).values)
    return X, y


X_train, y_train = to_tensors(train)
X_validation, y_validation = to_tensors(validation)

print(X_train.shape, y_train.shape)

91244 32491 37197
torch.Size([91244, 384]) torch.Size([91244])


In [3]:
mlflow.set_experiment("complaint-routing")

split_tags = {
    "train": "2023-01-01 to 2023-09-30",
    "validation": "2023-10-01 to 2023-11-30",
    "test": "2023-12-01 onwards",
    "dataset": "CFPB complaints, Hugging Face mirror to 2024-02-09",
}

record = """Model:
TF-IDF (words and word pairs, min_df 3, 300,000 features, sublinear tf)
+ logistic regression, C=4, max_iter 2000

Features:
the complaint narrative only

Split:
train 2023-01-01 to 2023-09-30, validation 2023-10-01 to 2023-11-30,
test 2023-12-01 onwards

Validation macro F1:
0.724

Test result:
0.870 accuracy, 0.717 macro F1
"""

with mlflow.start_run(run_name="tfidf-lr-C4") as run:
    mlflow.log_params(
        {
            "ngram_range": (1, 2),
            "min_df": 3,
            "max_features": 300_000,
            "sublinear_tf": True,
            "C": 4,
            "max_iter": 2000,
            "features": "narrative",
        }
    )

    benchmark = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=3,
                max_features=300_000,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                C=4,
                max_iter=2000,
            ),
        ),
    ])

    benchmark.fit(
        train["Consumer complaint narrative"],
        train["routing_team"],
    )

    predictions = benchmark.predict(validation["Consumer complaint narrative"])
    accuracy, macro_f1 = scores(validation["routing_team"], predictions)
    print(accuracy, macro_f1)

    mlflow.log_metrics(
        {
            "validation_accuracy": accuracy,
            "validation_macro_f1": macro_f1,
            "test_accuracy": 0.870,
            "test_macro_f1": 0.717,
        }
    )
    mlflow.set_tags(split_tags)
    mlflow.log_text(record, "record.txt")

    fig, ax = plt.subplots(figsize=(8, 8))
    ConfusionMatrixDisplay.from_predictions(
        validation["routing_team"],
        predictions,
        labels=teams,
        xticks_rotation=45,
        colorbar=False,
        ax=ax,
    )
    fig.tight_layout()
    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.close(fig)

    model_info = mlflow.sklearn.log_model(
        benchmark,
        name="model",
        serialization_format="pickle",
        input_example=np.array(
            [
                "My mortgage servicer added fees to my escrow account.",
                "A debt collector keeps calling me about a debt I do not owe.",
            ]
        ),
    )

print(run.info.run_id)
print(model_info.model_uri)

0.891 0.722


2026/10/16 15:53:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


98ac8c57882344ba8542f0d14e645181
models:/m-989dcc3c3d544d7f8108b7a99bae78f3


In [4]:
for C in [1, 10]:
    with mlflow.start_run(run_name=f"tfidf-lr-C{C}"):
        mlflow.log_params(
            {
                "ngram_range": (1, 2),
                "min_df": 3,
                "max_features": 300_000,
                "sublinear_tf": True,
                "C": C,
                "max_iter": 2000,
                "features": "narrative",
            }
        )

        model = Pipeline([
            (
                "tfidf",
                TfidfVectorizer(
                    ngram_range=(1, 2),
                    min_df=3,
                    max_features=300_000,
                    sublinear_tf=True,
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    C=C,
                    max_iter=2000,
                ),
            ),
        ])

        model.fit(
            train["Consumer complaint narrative"],
            train["routing_team"],
        )

        predictions = model.predict(validation["Consumer complaint narrative"])
        accuracy, macro_f1 = scores(validation["routing_team"], predictions)
        print(C, accuracy, macro_f1)

        mlflow.log_metrics(
            {
                "validation_accuracy": accuracy,
                "validation_macro_f1": macro_f1,
            }
        )
        mlflow.set_tags(split_tags)

        mlflow.sklearn.log_model(
            model,
            name="model",
            serialization_format="pickle",
            input_example=np.array(
                [
                    "My mortgage servicer added fees to my escrow account.",
                    "A debt collector keeps calling me about a debt I do not owe.",
                ]
            ),
        )

2026/10/16 15:56:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


1 0.884 0.679


10 0.892 0.73


2026/10/16 16:03:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [5]:
def validation_macro_f1(network):
    network.eval()
    with torch.no_grad():
        predictions = network(X_validation).argmax(dim=1)
    return f1_score(
        y_validation.numpy(),
        predictions.numpy(),
        average="macro",
    )


def fit(network, epochs):
    optimiser = torch.optim.Adam(network.parameters(), lr=0.001)
    loss_function = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        network.train()
        order = torch.randperm(len(X_train))
        total_loss = 0.0

        for start in range(0, len(order), 256):
            batch = order[start : start + 256]
            optimiser.zero_grad()
            loss = loss_function(network(X_train[batch]), y_train[batch])
            loss.backward()
            optimiser.step()
            total_loss += loss.item() * len(batch)

        training_loss = total_loss / len(order)
        validation_f1 = validation_macro_f1(network)

        print(
            f"epoch {epoch:2d}",
            f" training loss {training_loss:.3f}",
            f" validation macro F1 {validation_f1:.3f}",
        )
        mlflow.log_metrics(
            {
                "training_loss": training_loss,
                "validation_macro_f1": validation_f1,
            },
            step=epoch,
        )


def log_network(name, hidden_layers, network, epochs):
    with mlflow.start_run(run_name=name):
        mlflow.log_params(
            {
                "hidden_layers": hidden_layers,
                "epochs": epochs,
                "learning_rate": 0.001,
                "batch_size": 256,
                "seed": 0,
            }
        )
        mlflow.set_tags(split_tags)

        print(name)
        fit(network, epochs)

        if name == "two-hidden-layers":
            mlflow.log_metrics({"test_accuracy": 0.865, "test_macro_f1": 0.704})

        mlflow.pytorch.log_model(
            network,
            name="model",
            input_example=X_validation[:2].numpy(),
        )


torch.manual_seed(0)
log_network("one-layer", [], nn.Linear(384, 10), 30)

torch.manual_seed(0)
log_network(
    "one-hidden-layer",
    [256],
    nn.Sequential(
        nn.Linear(384, 256),
        nn.ReLU(),
        nn.Linear(256, 10),
    ),
    30,
)

torch.manual_seed(0)
log_network(
    "two-hidden-layers",
    [512, 256],
    nn.Sequential(
        nn.Linear(384, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 10),
    ),
    5,
)

one-layer
epoch  1  training loss 1.134  validation macro F1 0.122
epoch  2  training loss 0.683  validation macro F1 0.279
epoch  3  training loss 0.568  validation macro F1 0.371
epoch  4  training loss 0.505  validation macro F1 0.454
epoch  5  training loss 0.466  validation macro F1 0.501
epoch  6  training loss 0.440  validation macro F1 0.539
epoch  7  training loss 0.421  validation macro F1 0.561
epoch  8  training loss 0.408  validation macro F1 0.588
epoch  9  training loss 0.397  validation macro F1 0.607
epoch 10  training loss 0.389  validation macro F1 0.625
epoch 11  training loss 0.383  validation macro F1 0.637
epoch 12  training loss 0.377  validation macro F1 0.649
epoch 13  training loss 0.373  validation macro F1 0.655
epoch 14  training loss 0.369  validation macro F1 0.660
epoch 15  training loss 0.366  validation macro F1 0.664
epoch 16  training loss 0.363  validation macro F1 0.666
epoch 17  training loss 0.360  validation macro F1 0.673
epoch 18  training lo

one-hidden-layer
epoch  1  training loss 0.618  validation macro F1 0.522
epoch  2  training loss 0.371  validation macro F1 0.640
epoch  3  training loss 0.344  validation macro F1 0.683
epoch  4  training loss 0.330  validation macro F1 0.704
epoch  5  training loss 0.319  validation macro F1 0.706
epoch  6  training loss 0.311  validation macro F1 0.706
epoch  7  training loss 0.305  validation macro F1 0.711
epoch  8  training loss 0.298  validation macro F1 0.714
epoch  9  training loss 0.292  validation macro F1 0.712
epoch 10  training loss 0.288  validation macro F1 0.720
epoch 11  training loss 0.282  validation macro F1 0.723
epoch 12  training loss 0.277  validation macro F1 0.719
epoch 13  training loss 0.272  validation macro F1 0.723
epoch 14  training loss 0.268  validation macro F1 0.716
epoch 15  training loss 0.264  validation macro F1 0.719
epoch 16  training loss 0.259  validation macro F1 0.720
epoch 17  training loss 0.255  validation macro F1 0.721
epoch 18  trai

two-hidden-layers
epoch  1  training loss 0.542  validation macro F1 0.600
epoch  2  training loss 0.342  validation macro F1 0.675
epoch  3  training loss 0.317  validation macro F1 0.708
epoch  4  training loss 0.301  validation macro F1 0.722
epoch  5  training loss 0.288  validation macro F1 0.723


In [8]:
with open("data/prepared_runs.json") as file:
    prepared_runs = json.load(file)

for prepared in prepared_runs:
    with mlflow.start_run(run_name=prepared["run_name"]):
        mlflow.log_params(prepared["params"])
        mlflow.log_metrics(prepared["metrics"])
        mlflow.set_tags(prepared["tags"])

        if "system_prompt" in prepared:
            mlflow.log_text(prepared["system_prompt"], "system_prompt.txt")
            mlflow.log_dict(prepared["schema"], "schema.json")

In [9]:
runs = mlflow.search_runs(
    experiment_names=["complaint-routing"],
    order_by=["metrics.validation_macro_f1 DESC"],
)

columns = [
    "tags.mlflow.runName",
    "params.C",
    "params.hidden_layers",
    "params.model",
    "metrics.validation_macro_f1",
    "metrics.test_macro_f1",
]

table = runs[columns].rename(columns=lambda name: name.split(".")[-1])

print(table.round(3).to_string(index=False))

          runName   C hidden_layers                   model  validation_macro_f1  test_macro_f1
     tfidf-lr-C10  10           NaN                     NaN                0.730            NaN
     tfidf-lr-C10  10           NaN                     NaN                0.730            NaN
   distilbert-20k NaN           NaN distilbert-base-uncased                0.726          0.703
two-hidden-layers NaN    [512, 256]                     NaN                0.723          0.704
      tfidf-lr-C4   4           NaN                     NaN                0.722          0.717
      tfidf-lr-C4   4           NaN                     NaN                0.722          0.717
 one-hidden-layer NaN         [256]                     NaN                0.717            NaN
        one-layer NaN            []                     NaN                0.692            NaN
      tfidf-lr-C1   1           NaN                     NaN                0.679            NaN
      tfidf-lr-C1   1           NaN     

In [10]:
version = mlflow.register_model(model_info.model_uri, "complaint-router")

print(version.version)

1


Successfully registered model 'complaint-router'.
Created version '1' of model 'complaint-router'.


In [11]:
version = mlflow.register_model("runs:/6a728f5ca6c94530bf411fcf9cc77ddc/model", "complaint-router")

Registered model 'complaint-router' already exists. Creating a new version of this model...
2026/10/16 16:15:37 WARNING mlflow.tracking._model_registry.fluent: Run with id 6a728f5ca6c94530bf411fcf9cc77ddc has no artifacts at artifact path 'model', registering model based on models:/m-fe234c03f5ba48fd876467f0d6d248cc instead
Created version '2' of model 'complaint-router'.


In [12]:
client = mlflow.MlflowClient()
client.set_registered_model_alias("complaint-router", "champion", "1")

champion = mlflow.sklearn.load_model("models:/complaint-router@champion")

print(champion.predict(["They repossessed my car."]))

['vehicle_loan']
